In [6]:
%load_ext sql

The sql extension is already loaded. To reload it, use:
  %reload_ext sql


##### Connecting to the Database

In [11]:
%sql sqlite:///SoftDevEmployees.db

'Connected: @SoftDevEmployees.db'

In [15]:
%config SqlMagic.style = '_DEPRECATED_DEFAULT'

### Task One: Creating the required tables

In [24]:
%%sql

DROP TABLE IF EXISTS Titles_2NF;
DROP TABLE IF EXISTS Employee_role_2NF;
DROP TABLE IF EXISTS Employee_Department_2NF;
DROP TABLE IF EXISTS Departments_2NF;
DROP TABLE IF EXISTS Roles_2NF;
DROP TABLE IF EXISTS Titles_2NF;
DROP TABLE IF EXISTS Employee_2NF;

CREATE TABLE Employee_2NF (
	EmployeeID INTEGER NOT NULL,
	Name VARCHAR,
	Surname VARCHAR,
	Salary REAL,
	OccupationBand VARCHAR,
	TitleID INTEGER,
	PRIMARY KEY (EmployeeID AUTOINCREMENT),
	FOREIGN KEY (TitleID) REFERENCES Titles_2NF(TitleID)
);

CREATE TABLE Titles_2NF (
	TitleID INTEGER NOT NULL,
	Title VARCHAR,
	PRIMARY KEY (TitleID AUTOINCREMENT)
);

CREATE TABLE Roles_2NF (
	RoleID INTEGER NOT NULL,
	Role VARCHAR,
	PRIMARY KEY (RoleID AUTOINCREMENT)
);

CREATE TABLE Departments_2NF (
	DepartmentID INTEGER NOT NULL,
	Department VARCHAR,
	PRIMARY KEY (DepartmentID AUTOINCREMENT)
);

CREATE TABLE Employee_Department_2NF (
	EmployeeID INTEGER NOT NULL,
	DEPARTMENTID INTEGER NOT NULL,
	PRIMARY KEY (EmployeeID, DEPARTMENTID),
	FOREIGN KEY(DEPARTMENTID) REFERENCES Departments_2NF(DepartmentID),
	FOREIGN KEY(EmployeeID) REFERENCES Employees_2NF (EmployeeID)
);

CREATE TABLE Employee_Role_2NF (
	EmployeeID INTEGER NOT NULL,
	RoleID INTEGER NOT NULL,
	PRIMARY KEY (EmployeeID, RoleID),
	FOREIGN KEY (EmployeeID) REFERENCES Employees_2NF(EmployeeID),
	FOREIGN KEY (RoleID) REFERENCES Roles_2NF(RoleID)
);

 * sqlite:///SoftDevEmployees.db
Done.
Done.
Done.
Done.
Done.
Done.
Done.
Done.
Done.
Done.
Done.
Done.
Done.


[]

### Task Two: Populating Titles_2NF, Roles_2NF, and Departments_2NF

In [25]:
%%sql

SELECT
    *
FROM
    Employees_1NF;

 * sqlite:///SoftDevEmployees.db
Done.


Name,Surname,Department,Role,Title,OccupationBand,Salary
André,gerber,Web Applications,Front-End Developer,Mrs,Junior,52357.0
Antoinette,Van Der Berg,Mobile Applications,UI/UX Developer,Dr,Junior,118731.0
Bronwyn,Swartz,Mobile Applications,UI/UX Developer,Miss,Graduate,34350.0
Christopher,Walker,Mobile Applications,Back-End Developer,Mr,Junior,122894.0
Christopher,Walker,Web Applications,Back-End Developer,Mr,Junior,122894.0
Christopher,Walker,"Web Applications, Mobile Applications",Back-End Developer,Mr,Junior,122894.0
Claire,Morris,Mobile Applications,Full-Stack Developer,Ms,Intern,36000.0
Contact,Xaba,Mobile Applications,UI/UX Developer,Dr,Mid-Level,85836.0
Danie,Campbell,Mobile Applications,Business Analyst,Mrs,Mid-Level,205621.0
Danie,Campbell,Web Applications,Business Analyst,Mrs,Mid-Level,205621.0


In [26]:
%%sql

INSERT INTO Titles_2NF (Title)
SELECT DISTINCT
    Title
FROM
    Employees_1NF;

 * sqlite:///SoftDevEmployees.db
7 rows affected.


[]

In [27]:
%%sql

SELECT
    *
FROM
    Titles_2NF;

 * sqlite:///SoftDevEmployees.db
Done.


TitleID,Title
1,Mrs
2,Dr
3,Miss
4,Mr
5,Ms
6,Prof
7,None


In [50]:
%%sql

INSERT INTO Roles_2NF (Role)
SELECT DISTINCT
    Role
FROM
    Employees_1NF;

 * sqlite:///SoftDevEmployees.db
13 rows affected.


[]

In [51]:
%%sql

SELECT
    *
FROM
    Roles_2NF;

 * sqlite:///SoftDevEmployees.db
Done.


RoleID,Role
1,Front-End Developer
2,UI/UX Developer
3,Back-End Developer
4,Full-Stack Developer
5,Business Analyst
6,Database Analyst
7,Systems Analyst
8,"Full-Stack Developer, Scrum Master"
9,Scrum Master
10,"Full-Stack Developer, Team Lead"


In [28]:
%%sql

INSERT INTO Departments_2NF (Department)
SELECT DISTINCT
    Department
FROM
    Employees_1NF;

 * sqlite:///SoftDevEmployees.db
4 rows affected.


[]

In [29]:
%%sql

SELECT
    *
FROM
    Departments_2NF;

 * sqlite:///SoftDevEmployees.db
Done.


DepartmentID,Department
1,Web Applications
2,Mobile Applications
3,"Web Applications, Mobile Applications"
4,"Web Applications,Mobile Applications"


### Task Three: Populating the Employees_2NF

In [30]:
%sql PRAGMA table_info(employees_2nf);

 * sqlite:///SoftDevEmployees.db
Done.


cid,name,type,notnull,dflt_value,pk
0,EmployeeID,INTEGER,1,None,1
1,Name,VARCHAR,0,None,0
2,Surname,VARCHAR,0,None,0
3,Salary,REAL,0,None,0
4,OccupationBand,VARCHAR,0,None,0
5,TitleID,INTEGER,0,None,0


In [31]:
%%sql

INSERT INTO Employees_2NF (
    Name,
    Surname,
    Salary,
    OccupationBand,
    TitleID
)
SELECT DISTINCT
    emp.Name,
    emp.Surname,
    emp.Salary,
    emp.OccupationBand,
    ttl.TitleID
FROM
    Employees_1NF AS emp
INNER JOIN
    Titles_2NF AS ttl
ON
    emp.Title = ttl.Title;

 * sqlite:///SoftDevEmployees.db
49 rows affected.


[]

In [33]:
%%sql

SELECT
    *
FROM
    Employees_2NF
LIMIT 10;

 * sqlite:///SoftDevEmployees.db
Done.


EmployeeID,Name,Surname,Salary,OccupationBand,TitleID
1,André,gerber,52357.0,Junior,1
2,Antoinette,Van Der Berg,118731.0,Junior,2
3,Bronwyn,Swartz,34350.0,Graduate,3
4,Christopher,Walker,122894.0,Junior,4
5,Claire,Morris,36000.0,Intern,5
6,Contact,Xaba,85836.0,Mid-Level,2
7,Danie,Campbell,205621.0,Mid-Level,1
8,Danie,davies,313491.0,Senior,2
9,Dirk,Banda,37601.0,Intern,4
10,Douglas,Sibanda,57353.0,Mid-Level,4


### Task Four: Populating Employee_Department_2NF and Employee_Role_2NF

In [48]:
%%sql

DELETE FROM Employee_Department_2NF;
DELETE FROM Employee_Role_2NF;

INSERT INTO Employee_Department_2NF (
    EmployeeID,
    DepartmentID
)
SELECT DISTINCT
    EMP2.EmployeeID,
    DPT.DepartmentID
FROM
    Employees_1NF AS EMP1
JOIN
    Employees_2NF AS EMP2
ON
    EMP1.Name = Emp2.Name AND EMP1.Surname = EMP2.Surname
JOIN
    Departments_2NF AS DPT
ON
    EMP1.Department = DPT.Department;

INSERT INTO Employee_Role_2NF (
    EmployeeID,
    RoleID
)
SELECT DISTINCT
	EMP2.EmployeeID,
	R.RoleID
FROM
	Employees_1NF AS EMP1
JOIN
	Employees_2NF AS EMP2
ON
	EMP1.Name = Emp2.Name
	AND
	EMP1.Surname = Emp2.Surname
JOIN
	Roles_2NF AS R
ON
	EMP1.Role = R.Role;

 * sqlite:///SoftDevEmployees.db
130 rows affected.
0 rows affected.
130 rows affected.
0 rows affected.


[]

In [49]:
%%sql

SELECT
    *
FROM
    Employee_Role_2NF;

 * sqlite:///SoftDevEmployees.db
Done.


EmployeeID,RoleID
